# Falcon — CS-WAM (Diffusion-Aware EOT + Spatial Consensus)
Clean production notebook for `feat/saliency-hybrid`.


In [64]:
!nvidia-smi

Mon Nov 16 19:22:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             17W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [65]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.environ['HF_HOME'] = '/content/drive/MyDrive/falcon_hf_cache'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [66]:
# Clone or update Falcon to feat/saliency-hybrid
import os
if not os.path.exists('/content/Falcon'):
    !git clone -b feat/saliency-hybrid https://github.com/SRDdev/Falcon.git /content/Falcon
else:
    %cd /content/Falcon
    !git fetch origin
    !git checkout feat/saliency-hybrid
    !git pull origin feat/saliency-hybrid

%cd /content/Falcon/research
!git log -1 --oneline
!pip install -q -r requirements.txt
!pytest -m "not gpu" -q

/content/Falcon
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Total 10 (delta 8), reused 10 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 2.65 KiB | 903.00 KiB/s, done.
From https://github.com/SRDdev/Falcon
   b6ba134..0e3bf04  feat/saliency-hybrid -> origin/feat/saliency-hybrid
D	research/experiments/gentrack/results/decay_fits.csv
D	research/experiments/gentrack/results/fpr.csv
D	research/experiments/gentrack/results/paired_tests.csv
D	research/experiments/gentrack/results/summary_edit.csv
D	research/experiments/gentrack/results/summary_wm.csv
D	research/experiments/guided_test/results/decay_fits.csv
D	research/experiments/guided_test/results/fpr.csv
D	research/experiments/guided_test/results/paired_tests.csv
D	research/experiments/guided_test/results/summary_edit.csv
D	research/experiments/guided_test/results/summary_wm.csv
D	research/experiments/pilot/results/decay_fits.csv
D	research/experiments/pilot/results/fpr.

In [67]:
# Setup WAM weights and Drive experiment symlink
!mkdir -p external_repos checkpoints/wam

if not os.path.exists('external_repos/watermark-anything'):
    !git clone --depth 1 https://github.com/facebookresearch/watermark-anything.git external_repos/watermark-anything

if not os.path.exists('checkpoints/wam/wam_mit.pth'):
    !wget -q -O checkpoints/wam/wam_mit.pth https://dl.fbaipublicfiles.com/watermark_anything/wam_mit.pth
    !cp external_repos/watermark-anything/checkpoints/params.json checkpoints/wam/
else:
    print('WAM checkpoint already present.')

!mkdir -p /content/drive/MyDrive/Falcon_Experiments
!rm -rf experiments
!ln -s /content/drive/MyDrive/Falcon_Experiments experiments


WAM checkpoint already present.


## Run CS-WAM (Diffusion-Aware EOT) Experiment
Re-embeds `cs_wam` with SD-1.5 VAE + latent noise EOT, runs 4 rounds of `ip2p` and `sdedit`, and evaluates all tracks.


In [68]:
# Remove previous cs_wam outputs so it re-embeds with Diffusion-Aware EOT
!rm -rf experiments/saliency_hybrid/images/_wm/cs_wam
!rm -rf experiments/saliency_hybrid/images/baseline/*/cs_wam
!rm -rf experiments/saliency_hybrid/results/rows/*cs_wam*

!python run_experiment.py --config configs/saliency_hybrid.yaml --no-image-metrics

2026-11-16 19:22:31,646 INFO httpx: HTTP Request: HEAD https://huggingface.co/datasets/UB-CVML-Group/PIE_Bench_pp/resolve/main/1_change_object_80/V1-00000-of-00001.parquet "HTTP/1.1 302 Found"
2026-11-16 19:22:31,648 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-11-16 19:22:31,967 INFO httpx: HTTP Request: HEAD https://huggingface.co/datasets/UB-CVML-Group/PIE_Bench_pp/resolve/main/2_add_object_80/V1-00000-of-00001.parquet "HTTP/1.1 302 Found"
2026-11-16 19:22:32,246 INFO httpx: HTTP Request: HEAD https://huggingface.co/datasets/UB-CVML-Group/PIE_Bench_pp/resolve/main/3_delete_object_80/V1-00000-of-00001.parquet "HTTP/1.1 302 Found"
2026-11-16 19:22:32,523 INFO httpx: HTTP Request: HEAD https://huggingface.co/datasets/UB-CVML-Group/PIE_Bench_pp/resolve/main/4_change_attribute_content_40/V1-00000-of-00001.parquet "HTTP/1.1 302 Found"
2026-11-16 19:22:32,7

## Results & Summary Tables


In [69]:
import pandas as pd

df = pd.read_csv('experiments/saliency_hybrid/results/rows.csv', dtype={'id': str})
wm = df[df.watermark != 'none']

print('=== DETECTION RATE (det_rate) ===')
display(wm.pivot_table(index=['editor', 'watermark'], columns='level', values='detected', aggfunc='mean').round(3))

print('\n=== BIT ACCURACY (bit_acc) ===')
display(wm.pivot_table(index=['editor', 'watermark'], columns='level', values='bit_acc', aggfunc='mean').round(3))

!python analysis/analyze.py --exp experiments/saliency_hybrid

=== DETECTION RATE (det_rate) ===


level                         0         1         2         3         4
editor watermark                                                       
ip2p   cs_wam               1.0  0.814815  0.592593   0.37037  0.333333
       peccavi_hybrid  0.888889  0.037037       0.0       0.0       0.0
       trustmark            1.0  0.592593  0.444444  0.259259  0.185185
       wam                  1.0  0.481481   0.37037  0.333333  0.259259
       wam_saliency         1.0  0.333333  0.185185  0.074074  0.111111
sdedit cs_wam               1.0       0.0       0.0       0.0       0.0
       peccavi_hybrid  0.888889       0.0       0.0       0.0       0.0
       trustmark            1.0       0.0       0.0       0.0       0.0
       wam                  1.0       0.0       0.0       0.0       0.0
       wam_saliency         1.0       0.0       0.0       0.0       0.0


=== BIT ACCURACY (bit_acc) ===


level                      0      1      2      3      4
editor watermark                                        
ip2p   cs_wam          0.994  0.904  0.803  0.685  0.645
       peccavi_hybrid  0.967  0.590  0.588  0.553  0.529
       trustmark       0.999  0.718  0.661  0.613  0.574
       wam             1.000  0.736  0.669  0.663  0.652
       wam_saliency    0.997  0.683  0.628  0.558  0.552
sdedit cs_wam          0.994  0.538  0.536  0.524  0.529
       peccavi_hybrid  0.967  0.508  0.502  0.495  0.510
       trustmark       0.999  0.519  0.484  0.497  0.501
       wam             1.000  0.512  0.507  0.501  0.510
       wam_saliency    0.997  0.491  0.502  0.520  0.503

level                              0     1     2     3     4
method   editor watermark                                   
baseline ip2p   cs_wam          1.00  0.81  0.59  0.37  0.33
                peccavi_hybrid  0.89  0.04  0.00  0.00  0.00
                trustmark       1.00  0.59  0.44  0.26  0.19
                wam             1.00  0.48  0.37  0.33  0.26
                wam_saliency    1.00  0.33  0.19  0.07  0.11
         sdedit cs_wam          1.00  0.00  0.00  0.00  0.00
                peccavi_hybrid  0.89  0.00  0.00  0.00  0.00
                trustmark       1.00  0.00  0.00  0.00  0.00
                wam             1.00  0.00  0.00  0.00  0.00
                wam_saliency    1.00  0.00  0.00  0.00  0.00
level                               0      1      2      3      4
method   editor watermark                                        
baseline ip2p   cs_wam          0.994  0.904  0.803  0.685  0.645
                peccavi_hybrid  0.967  0.590  0.588  0.553  0.529
    

## Visual Grid Inspector


In [70]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

df = pd.read_csv('experiments/saliency_hybrid/results/rows.csv', dtype={'id': str})

def burn_hud_on_image(img, lvl, det, bit_acc, prompt, watermark):
    img = img.convert('RGBA')
    overlay = Image.new('RGBA', img.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    W, H = img.size
    draw.rectangle([(0, 0), (W, 65)], fill=(0, 0, 0, 185))
    draw.rectangle([(0, H - 35), (W, H)], fill=(0, 0, 0, 200))
    border_color = (76, 175, 80, 255) if det else (244, 67, 54, 255)
    draw.rectangle([(0, 0), (W - 1, H - 1)], outline=border_color, width=4)
    status_str = 'DETECTED [PASS]' if det else 'DESTROYED [FAIL]'
    draw.text((12, 10), f'Round L{lvl}: {status_str}', fill=border_color)
    draw.text((12, 34), f'Bit Acc: {bit_acc:.1f}%', fill=(255, 255, 255, 255))
    prompt_str = f'"{prompt[:55]}..."' if len(str(prompt)) > 55 else f'"{prompt}"'
    draw.text((12, H - 25), prompt_str, fill=(220, 220, 220, 255))
    return Image.alpha_composite(img, overlay).convert('RGB')

def show_watermark_grid(image_id='111000000000', editor='ip2p', watermark='cs_wam'):
    fig, axes = plt.subplots(1, 5, figsize=(24, 6))
    fig.patch.set_facecolor('#121212')
    sub_df = df[(df.id == str(image_id)) & (df.editor == editor) & (df.watermark == watermark)].set_index('level')
    for lvl in range(5):
        ax = axes[lvl]
        p = f'experiments/saliency_hybrid/images/_wm/{watermark}/{image_id}.png' if lvl == 0 else f'experiments/saliency_hybrid/images/baseline/{editor}/{watermark}/{image_id}/L{lvl}.png'
        if os.path.exists(p) and lvl in sub_df.index:
            img = Image.open(p).convert('RGB')
            row = sub_df.loc[lvl]
            prompt = row['instruction'] if lvl > 0 and pd.notna(row['instruction']) else 'Original Watermarked L0'
            ax.imshow(burn_hud_on_image(img, lvl, bool(row['detected']), float(row['bit_acc']) * 100, prompt, watermark))
        else:
            ax.imshow(Image.new('RGB', (512, 512), (30, 30, 30)))
            ax.text(256, 256, 'Not Found', color='white', ha='center')
        ax.axis('off')
    plt.suptitle(f'{watermark.upper()} under {editor.upper()} (Sample: {image_id})', color='white', fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()

show_watermark_grid(image_id='111000000000', editor='ip2p', watermark='cs_wam')
show_watermark_grid(image_id='111000000000', editor='ip2p', watermark='wam')


Output hidden; open in https://colab.research.google.com to view.